# Solución para el Problema del Código 2FA

Ahora que el sistema solicita la verificación 2FA pero no sabes dónde encontrar el código, aquí está la solución completa:

## 1. ¿Dónde está el código 2FA?

Cuando activaste 2FA durante el registro, el backend generó un **secreto 2FA** que se almacenó en la base de datos. Este secreto se usa para generar códigos temporales en tu aplicación autenticadora.

Para obtenerlo:

```bash
# Consulta el secreto en la base de datos
docker exec -it sqlite sqlite3 /var/lib/sqlite/sqlite.db "SELECT username, two_factor_secret FROM users WHERE username = 'tu_usuario';"
```

## 2. Cómo configurar tu aplicación autenticadora

Necesitas una aplicación como:
- Google Authenticator (Android/iOS)
- Microsoft Authenticator
- Authy
- FreeOTP

**Pasos para configurar:**

1. Abre tu aplicación autenticadora
2. Selecciona "Añadir cuenta" o "Escanear código QR"
3. Si tienes el QR code del registro:
   - Escanéalo con la app
4. Si no tienes QR:
   - Elige "Introducir clave manualmente"
   - Ingresa el `two_factor_secret` de la base de datos
   - Establece "Tipo" como TOTP (Time-based)
   - Normalmente usa 6 dígitos y 30 segundos de validez

## 3. Solución temporal para desarrollo

Si estás en entorno de desarrollo y necesitas generar códigos:

```bash
# Instala speakeasy (si no lo tienes)
npm install -g speakeasy

# Genera código actual (usa el two_factor_secret de la BD)
speakeasy totp --secret "TU_SECRETO_DE_BD" --encoding base32
```

## 4. Mejora el sistema para producción

Para hacer el sistema más usable, modifica el endpoint de registro para que:

1. Envíe el QR code por email (en producción)
2. Muestre el QR code en pantalla (en desarrollo)

**Ejemplo de modificación en `userRouter.js`:**

```javascript
if (enable2FA) {
    const secret = speakeasy.generateSecret({
        length: 20,
        name: `PongApp:${email}`,
        issuer: 'PongApp'
    });
    
    // Guardar solo el secreto
    twoFactorSecret = secret.base32;
    
    // En desarrollo: mostrar QR en consola
    if (process.env.NODE_ENV !== 'production') {
        QRCode.toString(secret.otpauth_url, {type: 'terminal'}, (err, url) => {
            console.log('\nEscanea este QR con tu app autenticadora:\n');
            console.log(url);
            console.log(`\nO ingresa manualmente: ${secret.base32}`);
        });
    }
    
    // En producción: enviar por email (implementar servicio de email)
}
```

## 5. Prueba completa del flujo

1. Registra nuevo usuario con 2FA:
```bash
curl -X POST http://localhost:3000/api/register \
-H "Content-Type: application/json" \
-d '{
    "username": "usuario2fa",
    "email": "usuario2fa@test.com",
    "password": "Password123!",
    "enable2FA": true
}'
```

2. Verifica en los logs del backend el QR code/secreto

3. Configura tu app autenticadora

4. Haz login:
```bash
curl -X POST http://localhost:3000/api/login \
-H "Content-Type: application/json" \
-d '{
    "username": "usuario2fa",
    "password": "Password123!"
}'
```

5. Verifica que pida 2FA y usa el código de tu app

## 6. Para administradores - Ver códigos existentes

```bash
# Ver todos los usuarios con 2FA activado
docker exec -it sqlite sqlite3 /var/lib/sqlite/sqlite.db "SELECT username, two_factor_secret FROM users WHERE two_factor_enabled = 1;"

# Ver tokens 2FA temporales
docker exec -it sqlite sqlite3 /var/lib/sqlite/sqlite.db "SELECT * FROM two_fa_tokens;"
```

Con estos pasos ya podrás completar el flujo de autenticación en dos factores correctamente. ¿Necesitas ayuda con alguna parte específica?

# Solución Integral para Verificación 2FA en Entornos Local y Producción

Basado en tu propuesta y considerando tus infraestructuras (Nginx, PHP, bases de datos), aquí tienes una solución profesional escalable:

## 1. Estrategia Recomendada (Híbrida)

**Para entorno local:**
- Mostrar QR directamente en pantalla después del registro
- Opción de ver código secreto para ingresar manualmente

**Para producción:**
- Enviar email con QR y código secreto
- Permitir selección de método de verificación (email/app)

## 2. Implementación Técnica

### Backend (`userRouter.js`):

```javascript
const nodemailer = require('nodemailer');
const QRCode = require('qrcode');

// Configuración para email (producción)
const transporter = nodemailer.createTransport({
  host: process.env.EMAIL_HOST,
  port: process.env.EMAIL_PORT,
  secure: true,
  auth: {
    user: process.env.EMAIL_USER,
    pass: process.env.EMAIL_PASS
  }
});

router.post('/register', async (req, res) => {
  const { username, email, password, enable2FA, authMethod = 'app' } = req.body;

  // ... validaciones ...

  try {
    let twoFactorData = null;
    if (enable2FA) {
      const secret = speakeasy.generateSecret({
        name: `PongApp:${email}`,
        issuer: 'PongApp'
      });

      twoFactorData = {
        secret: secret.base32,
        authMethod, // 'app' o 'email'
        enabled: true
      };

      // Generar QR Code como Data URL
      const qrCode = await QRCode.toDataURL(secret.otpauth_url);

      if (process.env.NODE_ENV === 'production') {
        // En producción: enviar por email
        await transporter.sendMail({
          from: '"PongApp" <no-reply@pongapp.com>',
          to: email,
          subject: 'Configura tu autenticación en dos pasos',
          html: `
            <h2>Configuración 2FA</h2>
            <p>Escanea este código con Google Authenticator:</p>
            <img src="${qrCode}" alt="QR Code"/>
            <p>O ingresa manualmente: <strong>${secret.base32}</strong></p>
            <p>Método seleccionado: ${authMethod}</p>
          `
        });
      } else {
        // En local: devolver QR en respuesta
        twoFactorData.qrCode = qrCode;
        twoFactorData.manualCode = secret.base32;
      }
    }

    // Guardar usuario en BD con twoFactorData
    // ...

    res.json({
      success: true,
      message: 'Usuario registrado',
      ...(process.env.NODE_ENV !== 'production' && enable2FA && { 
        twoFactorSetup: twoFactorData 
      })
    });

  } catch (error) {
    // ... manejo de errores
  }
});
```

### Frontend (registro.tsx):

```typescript
const Register = () => {
  const [authMethod, setAuthMethod] = useState<'app'|'email'>('app');
  const [twoFactorData, setTwoFactorData] = useState(null);

  // Mostrar QR después de registro exitoso
  if (twoFactorData) {
    return (
      <div className="verification-modal">
        <h3>Configura tu 2FA</h3>
        <img src={twoFactorData.qrCode} alt="QR Code"/>
        <p>Código manual: {twoFactorData.manualCode}</p>
        <button onClick={() => navigate('/login')}>Continuar</button>
      </div>
    );
  }

  return (
    <form onSubmit={handleSubmit}>
      {/* Campos del formulario */}
      
      <div className="auth-method-selector">
        <label>
          <input 
            type="radio" 
            checked={authMethod === 'app'}
            onChange={() => setAuthMethod('app')}
          />
          Usar app autenticadora (Google/Microsoft Authenticator)
        </label>
        
        <label>
          <input
            type="radio"
            checked={authMethod === 'email'}
            onChange={() => setAuthMethod('email')}
          />
          Recibir códigos por email
        </label>
      </div>

      <button type="submit">Registrarse</button>
    </form>
  );
};
```

## 3. Flujo de Verificación por Email

Para el método de email, modifica el endpoint de login:

```javascript
router.post('/login', async (req, res) => {
  // ... lógica normal de login ...

  if (user.two_factor_enabled) {
    if (user.auth_method === 'email') {
      // Generar código de 6 dígitos
      const emailCode = Math.floor(100000 + Math.random() * 900000);
      
      // Guardar código en BD con expiración
      await db.run(
        `INSERT INTO two_fa_codes 
        (user_id, code, expires_at) 
        VALUES (?, ?, datetime('now', '+10 minutes'))`,
        [user.id, emailCode]
      );

      // Enviar por email
      if (process.env.NODE_ENV === 'production') {
        await transporter.sendMail({
          to: user.email,
          subject: 'Tu código de verificación PongApp',
          text: `Tu código es: ${emailCode}`
        });
      } else {
        // En local: devolver código en respuesta
        return res.status(202).json({
          requires2FA: true,
          verificationMethod: 'email',
          debugCode: emailCode, // Solo en desarrollo
          message: 'Código enviado por email'
        });
      }
    } else {
      // Lógica normal para app autenticadora
    }
  }
});
```

## 4. Prácticas Recomendadas

1. **Entorno Local**:
   - Mostrar QR/código directamente en pantalla
   - Permitir copiar código con un click
   - Incluir botón "Simular recepción de email" para testing

2. **Producción**:
   - Usar servicio de email profesional (SendGrid, Mailgun, etc.)
   - Implementar reintentos de envío
   - Registrar en logs los envíos fallidos

3. **Seguridad**:
   - Limitar intentos de verificación (3-5 intentos)
   - Invalidar códigos después de 10-15 minutos
   - No mostrar códigos completos en frontend en producción

4. **UX**:
   - Permitir cambiar método de autenticación
   - Opción "No recibí el código" con reenvío
   - Recordar dispositivo por 30 días (opcional)

## 5. Implementación Adicional Recomendada

Crea una ruta para reenviar códigos:

```javascript
router.post('/resend-2fa', authMiddleware, async (req, res) => {
  const { userId } = req.user;
  const user = await db.get('SELECT * FROM users WHERE id = ?', [userId]);
  
  if (user.two_factor_enabled) {
    if (user.auth_method === 'email') {
      // Misma lógica que en login
    } else {
      // Regenerar QR si es necesario
    }
  }
});
```

Esta solución ofrece flexibilidad para ambos entornos manteniendo altos estándares de seguridad y usabilidad. ¿Necesitas que desarrolle más alguna parte específica?

# Análisis y Recomendaciones para la Implementación de 2FA y Google Sign-In

Basado en los requerimientos del proyecto, aquí tienes un análisis estructurado y recomendaciones para implementar correctamente ambos sistemas de autenticación:

## 1. Compatibilidad entre Google Sign-In y 2FA

**¿Se pueden implementar juntos?**  
✅ **Sí**, pero con consideraciones importantes:

- **Google Sign-In** es un método de autenticación primario (reemplaza usuario/contraseña)
- **2FA** es una capa adicional de seguridad que requiere un segundo factor

### Implementación recomendada:

1. **Flujo para usuarios con Google Sign-In**:
   - Autenticación inicial con Google
   - Si el usuario tiene 2FA habilitado:
     - Redirigir a página de verificación 2FA
     - Usar código de app autenticadora (no email/SMS por seguridad)

2. **Flujo para usuarios estándar**:
   - Login con usuario/contraseña
   - Verificación 2FA si está habilitado (app/email/SMS)

## 2. Diagrama de Flujo de Autenticación Combinado

```
                       [Inicio Sesión]
                             |
              +--------------+--------------+
              |                             |
      [Google Sign-In]             [Credenciales Locales]
              |                             |
       (Obtener token ID)           (Verificar usuario/contraseña)
              |                             |
      +--------+--------+                   |
      |                 |                   |
[Primer login?]   [Usuario existente]       |
      |                 |                   |
(Registro automático) (Verificar 2FA) <-----+
                             |
                      [Método 2FA Configurado]
                             |
              +--------------+--------------+
              |              |             |
        [App Auth]     [Email]       [SMS] (no recomendado)
              |              |             |
        (Ingresar código) (Ingresar código)
                             |
                      [Generar JWT]
                             |
                      [Acceso concedido]
```

## 3. Recomendaciones Técnicas Clave

### Para Google Sign-In:
1. **Configuración en Google Cloud Console**:
   - Crear credenciales OAuth 2.0
   - Establecer URI de redireccionamiento válidos
   - Habilitar scope `email profile openid`

2. **Implementación en backend**:
```javascript
// Ruta para autenticación con Google
router.post('/auth/google', async (req, res) => {
  const { tokenId } = req.body;
  
  try {
    const ticket = await client.verifyIdToken({
      idToken: tokenId,
      audience: process.env.GOOGLE_CLIENT_ID
    });
    
    const payload = ticket.getPayload();
    
    // Buscar o crear usuario
    let user = await findOrCreateUserFromGoogle(payload);
    
    // Verificar si requiere 2FA
    if (user.two_factor_enabled) {
      const tempToken = generateTempToken();
      return res.status(202).json({
        requires2FA: true,
        tempToken,
        userId: user.id
      });
    }
    
    // Generar JWT si no necesita 2FA
    const token = generateJWT(user);
    res.json({ token });
    
  } catch (error) {
    res.status(401).json({ error: 'Autenticación fallida' });
  }
});
```

### Para 2FA:
1. **Opciones de implementación** (ordenadas por seguridad):
   1. **App Authenticator** (Google/Microsoft Auth) - Más seguro
   2. **Email** - Medio seguro (depende de seguridad del email)
   3. **SMS** - Menos seguro (evitar si es posible)

2. **Ejemplo de middleware de verificación**:
```javascript
const verify2FA = async (req, res, next) => {
  if (!req.user.two_factor_enabled) return next();
  
  const { code } = req.body;
  const isValid = await verify2FACode(req.user.id, code);
  
  if (!isValid) {
    return res.status(401).json({ error: 'Código 2FA inválido' });
  }
  
  next();
};

// Uso en rutas protegidas
router.post('/protected-route', authMiddleware, verify2FA, (req, res) => {
  // Acceso concedido
});
```

## 4. Consideraciones de Seguridad Críticas

1. **Para Google Sign-In**:
   - Validar siempre el `tokenId` en el backend
   - Verificar el `audience` (tu CLIENT_ID)
   - Nunca aceptar tokens directamente en el frontend

2. **Para 2FA**:
   - Limitar intentos fallidos (3-5 intentos)
   - Usar códigos de al menos 6 dígitos
   - Invalidar códigos después de 3-5 minutos
   - No permitir reutilización de códigos

3. **Para JWT**:
   - Usar tiempos de expiración cortos (15-30 mins para access token)
   - Implementar refresh tokens con expiración (7 días)
   - Firmar con algoritmo fuerte (HS256 o RS256)

## 5. Integración con los Requerimientos del Proyecto

### Cumplimiento de User Management:
- **Google Sign-In** cubre:
  - Suscripción segura (registro automático)
  - Login seguro
- **2FA** añade:
  - Capa adicional de seguridad para cuentas privilegiadas

### Cumplimiento de Remote Authentication:
- Google Sign-In es el sistema remoto
- 2FA/JWT proveen la seguridad local

### Recomendación final:
Implementa primero Google Sign-In como método principal, luego añade 2FA como opción configurable en el perfil de usuario, con preferencia a app authenticator sobre email/SMS.

¿Necesitas que desarrolle más algún aspecto específico de la implementación?